# LC 215 — Kth Largest Element in an Array
**Day 52 | Heap Review | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> You don't need to sort everything.
Keep a min-heap of size <em>k</em> — the smallest of the k largest
always sits at the top, ready to return in O(1).
</div>

## Official Problem Statement

Given an integer array `nums` and an integer `k`, return the **k-th
largest** element in the array.

Note that it is the k-th largest element in **sorted order**, not
the k-th distinct element.

Can you solve it without sorting?

**Constraints:**
- `1 <= k <= nums.length <= 10^5`
- `-10^4 <= nums[i] <= 10^4`

## What This Is Actually Asking

If you sorted the array descending, which element is at index k-1?
We want that value without paying the full O(n log n) sort cost.
The k-th largest is also the (n-k+1)-th smallest — that framing
hints at a min-heap of bounded size k.
Once the heap reaches size k, anything smaller than the heap top
can never be a top-k candidate, so we discard it immediately.

## Walk Through an Example by Hand

```
nums = [3, 2, 1, 5, 6, 4], k = 2

Push 3  → heap: [3]
Push 2  → heap: [2, 3]
Push 1  → heap: [1, 3, 2]  len=3 > k=2 → pop 1 → [2, 3]
Push 5  → heap: [2, 3, 5]  len=3 > k=2 → pop 2 → [3, 5]
Push 6  → heap: [3, 5, 6]  len=3 > k=2 → pop 3 → [5, 6]
Push 4  → heap: [4, 6, 5]  len=3 > k=2 → pop 4 → [5, 6]

heap[0] = 5  ✓  (2nd largest of [1,2,3,4,5,6])
```

## The Picture

```
Stream of nums ──► MIN-HEAP (size capped at k)

  [3, 2, 1, 5, 6, 4]   k = 2
   │
   ▼  after each push + optional pop:

   heap (min at top)    discard
   ─────────────────    ───────
   [3]                  —
   [2, 3]               —
   [2, 3]  ◄── pop 1    1
   [3, 5]  ◄── pop 2    2
   [5, 6]  ◄── pop 3    3
   [5, 6]  ◄── pop 4    4

   answer = heap[0] = 5
   ^^^^^^^^^^^^^^^^
   smallest of the k largest  =  kth largest
```

## When To Use This Pattern

- When asked for the **k-th largest / smallest** from a stream or
  array, think **bounded min-heap of size k**.
- When you need **top-k frequent** elements, think heap + Counter.
- When sorting the full array is too expensive (n large, k small),
  think heap for O(n log k) instead of O(n log n).
- When data arrives as a stream with no random access, think heap
  (you process one element at a time).
- When the problem says "without sorting", think heap.

## The Approach

Maintain a min-heap whose size never exceeds k.
For every number, push it onto the heap; if the heap grows beyond k,
pop the minimum — that number is provably not in the top k.
After processing all numbers the heap holds exactly the k largest
values, and the minimum of that group (heap[0]) is the answer.

In [ ]:
import heapq
from typing import List

In [ ]:
def test_harness(func):
    cases = [
        ([3, 2, 1, 5, 6, 4], 2, 5),
        ([3, 2, 3, 1, 2, 4, 5, 5, 6], 4, 4),
        ([1], 1, 1),
        ([7, 6, 5, 4, 3, 2, 1], 3, 5),
        ([-1, -2, -3, -4, -5], 2, -2),
    ]
    passed = 0
    for nums, k, expected in cases:
        result = func(nums[:], k)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"{status} | nums={nums}, k={k} "
            f"| expected={expected}, got={result}"
        )
    print(f"\n{passed}/{len(cases)} tests passed.")

In [ ]:
def find_kth_largest(nums: List[int], k: int) -> int:
    """
    Return the k-th largest element in nums.

    Strategy: min-heap capped at size k.
      - Push each num onto the heap.
      - If heap size exceeds k, pop the minimum.
      - heap[0] is the answer.

    Time : O(n log k)
    Space: O(k)

    Args:
        nums: list of integers
        k   : which largest to find (1-indexed)

    Returns:
        The k-th largest integer.
    """
    heap = []
    for num in nums:
        print(f"  push {num}")
        heapq.heappush(heap, num)
        if len(heap) > k:
            removed = heapq.heappop(heap)
            print(f"  pop  {removed} | heap={heap}")
    print(f"Final heap: {heap}")
    pass  # TODO: return heap[0]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(find_kth_largest)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Sort | O(n log n) | O(1) | Simple but wasteful |
| Min-heap size k | O(n log k) | O(k) | Optimal for streams |
| QuickSelect | O(n) avg | O(1) | Best avg, O(n²) worst |

## Real World Connection

At **Citi**, risk dashboards must surface the top-k largest
P&L swings across thousands of positions in near real-time.
Sorting the full position book every tick is too slow; a bounded
min-heap processes each new trade in O(log k) and always holds
the current top-k ready to display.
On **AWS** EMR, the same pattern appears in distributed top-k
aggregations: each mapper maintains a local heap of size k, and
the reducer merges at most k×mappers elements — far cheaper than
a full sort-shuffle.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra